处理职官条目分类目录的 OCR 结果，其中包含
  * 主标题，副标题，三级标题
  * 文本部分，包含主名和别称
  * 错误提取为表格的数据
  * 其他错误

基本数据结构
  * type: title, text | h1, h2, h3, name, surname
  * text: string
  * page: number
  * bbox: 保留位置信息

理想数据结构：编 - 大类 - (小类) - 名称 - 别名


In [ ]:
import json
with open("../../data/ocr-results/MinerU_宋代官制辞典-职官条目分类目录__20260108090208.json", "r", encoding="utf-8") as f:
  ocr_result = json.load(f)


In [2]:
"""误提取的表格数据"""
raw_tables = [
  "<table><tr><td>枢轴</td><td>113</td><td>签书院事</td><td>115</td><td>枢密院在京房</td><td>117</td></tr><tr><td>枢长</td><td>113</td><td>签枢</td><td>115</td><td>枢密院教阅房</td><td>117</td></tr><tr><td>枢密太尉</td><td>113</td><td>枢密</td><td>115</td><td>枢密院广西房</td><td>117</td></tr><tr><td>枢使</td><td>113</td><td>同签书枢密院事</td><td>115</td><td>枢密院兵籍房</td><td>117</td></tr><tr><td>枢密副使</td><td>113</td><td>同签书枢密院</td><td>115</td><td>枢密院民兵房</td><td>117</td></tr><tr><td>枢副</td><td>113</td><td>同签书</td><td>115</td><td>枢密院知杂房</td><td>117</td></tr><tr><td>副枢</td><td>113</td><td>同签书院事</td><td>115</td><td>枢密院支马房</td><td>117</td></tr><tr><td>副密</td><td>113</td><td>同签枢</td><td>115</td><td>枢密院小吏房</td><td>117</td></tr><tr><td>枢近</td><td>113</td><td>枢密</td><td>115</td><td>枢密院二十五房</td><td>117</td></tr><tr><td>贰枢廷</td><td>113</td><td>领枢密院事</td><td>115</td><td>枢密院机速房</td><td>118</td></tr><tr><td>贰枢</td><td>113</td><td>领枢密院</td><td>115</td><td>枢密院写宣房</td><td>118</td></tr><tr><td>判枢密院事</td><td>113</td><td>领院事</td><td>115</td><td>枢密院编修时政记房</td><td>118</td></tr><tr><td>判院事</td><td>113</td><td>领院</td><td>115</td><td>政记房</td><td>118</td></tr><tr><td>判枢密院</td><td>113</td><td>权领枢密院事</td><td>115</td><td>正记房</td><td>118</td></tr><tr><td>知枢密院事</td><td>113</td><td>权领枢密院</td><td>115</td><td>枢密院干办官</td><td>118</td></tr><tr><td>知枢密院</td><td>114</td><td>权同发遣枢密院事</td><td>115</td><td>干办官</td><td>118</td></tr><tr><td>知密院</td><td>114</td><td>枢密院承旨司</td><td>115</td><td>枢密院计议官</td><td>118</td></tr><tr><td>知院</td><td>114</td><td>承旨司</td><td>116</td><td>计议官</td><td>118</td></tr><tr><td>知院</td><td>114</td><td>枢密院承旨</td><td>116</td><td>枢密院干办官</td><td>118</td></tr><tr><td>枢密院事</td><td>114</td><td>枢密承旨</td><td>116</td><td>枢密院检详所</td><td>118</td></tr><tr><td>枢密</td><td>114</td><td>枢密院都承旨</td><td>116</td><td>检详所</td><td>118</td></tr><tr><td>长本兵</td><td>114</td><td>枢密都承旨</td><td>116</td><td>枢密院检详诸房文字</td><td>118</td></tr><tr><td>同知枢密院事</td><td>114</td><td>都承旨</td><td>116</td><td>枢密院检详</td><td>119</td></tr><tr><td>同知院事</td><td>114</td><td>都承</td><td>116</td><td>枢密院检详文字</td><td>119</td></tr><tr><td>同知</td><td>114</td><td>权枢密院都承旨</td><td>116</td><td>检详</td><td>119</td></tr><tr><td>同知枢</td><td>114</td><td>枢密院副都承旨</td><td>116</td><td>检详官</td><td>119</td></tr><tr><td>同知枢密院</td><td>114</td><td>枢密副都承旨</td><td>116</td><td>检详枢密院诸房文字</td><td>119</td></tr><tr><td>同知枢密</td><td>114</td><td>副都承旨</td><td>116</td><td>枢密院编修</td><td>119</td></tr><tr><td>枢密</td><td>114</td><td>枢密院副承旨</td><td>116</td><td>编修</td><td>119</td></tr><tr><td>贰枢</td><td>114</td><td>枢密院副承旨</td><td>116</td><td>枢密院编修司</td><td>119</td></tr><tr><td>贰枢笺</td><td>114</td><td>密院副承旨</td><td>116</td><td>枢属</td><td>119</td></tr><tr><td>枢副</td><td>114</td><td>枢密院诸房副承旨</td><td>116</td><td>枢掾</td><td>119</td></tr><tr><td>签署枢密院事</td><td>114</td><td>枢密院五房</td><td>116</td><td>枢密院制置兵马司</td><td>119</td></tr><tr><td>签署院事</td><td>114</td><td>枢密院兵房</td><td>116</td><td>兵马司</td><td>119</td></tr><tr><td>签署</td><td>114</td><td>枢密院吏房</td><td>116</td><td>枢密院讲议司</td><td>119</td></tr><tr><td>签书枢密院事</td><td>114</td><td>枢密院户房</td><td>116</td><td>银台司</td><td>119</td></tr><tr><td>同签署枢密院事</td><td>114</td><td>枢密院礼房</td><td>117</td><td>银台</td><td>120</td></tr><tr><td>签书枢密院事</td><td>115</td><td>枢密院刑房</td><td>117</td><td>银司</td><td>120</td></tr><tr><td>签书枢密院</td><td>115</td><td>枢密院十二房</td><td>117</td><td>知通进、银台司公事</td><td>120</td></tr><tr><td>签书枢密院事</td><td>115</td><td>枢密院北面房</td><td>117</td><td>点检银台通进司公事</td><td>120</td></tr><tr><td>签书枢密</td><td>115</td><td>枢密院河西房</td><td>117</td><td>知通进、银台司兼门</td><td></td></tr><tr><td>签书</td><td>115</td><td>枢密院支差房</td><td>117</td><td>下封驳公事</td><td>120</td></tr></table>",
  "<table><tr><td>史馆 162</td><td>编校集贤院书籍 165</td><td>寓职 168</td></tr><tr><td>史馆编修 162</td><td>编校集贤书籍 165</td><td>集贤殿修撰 168</td></tr><tr><td>编修 163</td><td>编校书籍 165</td><td>集英殿修撰 168</td></tr><tr><td>史馆检讨 163</td><td>校书集贤 165</td><td>集英 168</td></tr><tr><td>检讨 163</td><td>秘阁 165</td><td>热撰 168</td></tr><tr><td>史讨 163</td><td>中秘 165</td><td>集撰 168</td></tr><tr><td>史馆校勘 163</td><td>东观 165</td><td>右文殿修撰 168</td></tr><tr><td>校勘 163</td><td>石渠 165</td><td>右文 168</td></tr><tr><td>史馆祗候 163</td><td>册府 165</td><td>冷撰 168</td></tr><tr><td>编校史馆书籍 163</td><td>秘府 165</td><td>殿撰 168</td></tr><tr><td>编校书籍 163</td><td>木天 165</td><td>秘阁修撰 168</td></tr><tr><td>编校官 163</td><td>直秘阁 166</td><td>秘撰 168</td></tr><tr><td>集贤院 163</td><td>秘阁 166</td><td>冷撰 168</td></tr><tr><td>集贤 163</td><td>直阁 166</td><td>旧馆阁 168</td></tr><tr><td>集贤院(殿)大学士 163</td><td>提辖秘阁供御图书 166</td><td>贴职修撰 169</td></tr><tr><td>集贤 164</td><td>判秘阁事 166</td><td>直龙图阁 169</td></tr><tr><td>集贤殿大学士 164</td><td>判阁 166</td><td>直天章阁 169</td></tr><tr><td>集贤殿直学士 164</td><td>判秘阁 166</td><td>直宝文阁 169</td></tr><tr><td>集贤院学士 164</td><td>秘阁校理 166</td><td>直显谟阁 169</td></tr><tr><td>集贤学士 164</td><td>校理 166</td><td>直徽猷阁 169</td></tr><tr><td>集贤殿修撰 164</td><td>秘教 166</td><td>直敷文阁 169</td></tr><tr><td>集贤修撰 164</td><td>监秘阁图书 166</td><td>直焕章阁 169</td></tr><tr><td>修撰 164</td><td>监图书官 166</td><td>直华文阁 169</td></tr><tr><td>判集贤院事 164</td><td>编校秘阁书籍 166</td><td>直宝谟阁 169</td></tr><tr><td>判院事 164</td><td>编校书籍 166</td><td>直宝章阁 169</td></tr><tr><td>判集贤院 164</td><td>编校官 167</td><td>直显文阁 169</td></tr><tr><td>直集贤院 164</td><td>馆阁编校书籍 167</td><td>直秘阁 169</td></tr><tr><td>直院 165</td><td>馆阁校勘 167</td><td>直睿思殿 169</td></tr><tr><td>集贤院校理 165</td><td>校勘官 167</td><td>直殿 169</td></tr><tr><td>集贤校理 165</td><td>馆阁读书 167</td><td>直阁 169</td></tr><tr><td>校理 165</td><td>贴职 167</td><td>直延阁 169</td></tr><tr><td>集校 165</td><td>阁职 168</td><td>睿思殿供奉 169</td></tr></table>",
  "<table><tr><td colspan=\"2\">一、三省门</td><td>省台 170\n三省长官 170\n两令 170\n三省都录事 170\n经抚房 170\n两省 170</td><td>北省 170\n掖署 170\n内两省 170\n大两省 171\n两省长官 171\n两省侍郎 171</td></tr><tr><td>三省</td><td>170</td><td>三省都录事</td><td>170</td></tr><tr><td>东府</td><td>170</td><td>经抚房</td><td>170</td></tr><tr><td>台省</td><td>170</td><td>两省</td><td>170</td></tr></table>",
  "<table><tr><td>国子监书库官 384</td><td>大学生 388</td><td>武学 390</td></tr><tr><td>监国子监公厨 384</td><td>无官御史 388</td><td>右学 391</td></tr><tr><td>国子监助教 385</td><td>大学外舍生 388</td><td>武学教授 391</td></tr><tr><td>国子学 385</td><td>太学外舍 388</td><td>教授 391</td></tr><tr><td>国学 385</td><td>外舍 388</td><td>武学传授 391</td></tr><tr><td>国子 385</td><td>大学内舍生 388</td><td>传授 391</td></tr><tr><td>学寮 385</td><td>内舍生 388</td><td>武学博士 391</td></tr><tr><td>厨库寮 385</td><td>内舍 388</td><td>武学博 391</td></tr><tr><td>知杂寮 385</td><td>太学上舍生 388</td><td>武博士 391</td></tr><tr><td>胥长 385</td><td>上舍 389</td><td>武学谕 391</td></tr><tr><td>胥史 385</td><td>走马上舍 389</td><td>学谕 391</td></tr><tr><td>胥佐 385</td><td>太学两优释褐人 389</td><td>武谕 392</td></tr><tr><td>贴书 385</td><td>两优释褐 389</td><td>谕 392</td></tr><tr><td>三京国子监 385</td><td>释褐状元 389</td><td>武学正 392</td></tr><tr><td>国学 385</td><td>太学三舍生 389</td><td>学正 392</td></tr><tr><td>西监 385</td><td>俊士 389</td><td>武学录 392</td></tr><tr><td>太学 386</td><td>辟雍 389</td><td>学录 392</td></tr><tr><td>大学 386</td><td>外学 389</td><td>司计 392</td></tr><tr><td>上库 386</td><td>辟雍大司成 389</td><td>武学生 392</td></tr><tr><td>贤关 386</td><td>辟雍司成 389</td><td>武选士 392</td></tr><tr><td>天子之学 386</td><td>大司成 389</td><td>武俊士 392</td></tr><tr><td>管勾太学公事 386</td><td>太学大司成 389</td><td>武士 392</td></tr><tr><td>同管勾太学公事 386</td><td>大司成 389</td><td>律学馆 392</td></tr><tr><td>权管勾太学公事 386</td><td>太学司成 390</td><td>律学助教 392</td></tr><tr><td>太学博士 386</td><td>辟雍司业 390</td><td>律学 392</td></tr><tr><td>太博 386</td><td>司业 390</td><td>律学教授 392</td></tr><tr><td>博士 386</td><td>辟雍丞 390</td><td>教授 392</td></tr><tr><td>大学博士 387</td><td>丞 390</td><td>律学博士 393</td></tr><tr><td>太学正 387</td><td>辟雍主簿 390</td><td>博士 393</td></tr><tr><td>学正 387</td><td>辟雍博士 390</td><td>律学正 393</td></tr><tr><td>正 387</td><td>博士 390</td><td>学正 393</td></tr><tr><td>大学正 387</td><td>辟雍正 390</td><td>律学录 393</td></tr><tr><td>职事学正 387</td><td>正 390</td><td>学录 393</td></tr><tr><td>太学录 387</td><td>辟雍录 390</td><td>律学生 393</td></tr><tr><td>学录 387</td><td>录 390</td><td>律学士 393</td></tr><tr><td>太学录事 387</td><td>命官正录 390</td><td>在京小学 393</td></tr><tr><td>录 387</td><td>辟雍直学 390</td><td>小学教谕 393</td></tr><tr><td>职事学录 387</td><td>命官直学 390</td><td>教谕 393</td></tr><tr><td>太学正录 387</td><td>学生直学 390</td><td>职事教谕 393</td></tr><tr><td>正录 387</td><td>贡士 390</td><td>小学录 393</td></tr><tr><td>太学助教 387</td><td>进士 390</td><td>小学学长 393</td></tr><tr><td>太学说书 388</td><td>文士 390</td><td>学长 394</td></tr></table>",
  "<table><tr><td>提举临安府洞霄宫</td><td>674</td></tr><tr><td>提举南京鸿庆宫</td><td>674</td></tr><tr><td>提点宫观</td><td>674</td></tr><tr><td>提点万寿观公事</td><td>674</td></tr><tr><td>提点佑神观公事</td><td>674</td></tr><tr><td>管勾宫观</td><td>674</td></tr><tr><td>管勾祥源观公事</td><td>674</td></tr><tr><td>管勾祥源观事</td><td>674</td></tr><tr><td>管勾兖州仙源县景灵宫</td><td></td></tr><tr><td>太极观公事</td><td>674</td></tr><tr><td>主管台州崇道观</td><td>675</td></tr><tr><td>管勾(主管)成都府玉局观</td><td>675</td></tr><tr><td>玉局</td><td>675</td></tr><tr><td>监岳庙</td><td>675</td></tr><tr><td>监庙</td><td>675</td></tr></table>",
  "<table><tr><td>岳庙</td><td>675</td></tr><tr><td>破格岳庙</td><td>675</td></tr><tr><td>义官</td><td>675</td></tr><tr><td>散官</td><td>675</td></tr><tr><td>散秩</td><td>675</td></tr><tr><td>节度副使</td><td>675</td></tr><tr><td>节度行军司马</td><td>675</td></tr><tr><td>节度司马</td><td>676</td></tr><tr><td>行军司马</td><td>676</td></tr><tr><td>军司马</td><td>676</td></tr><tr><td>典午</td><td>676</td></tr><tr><td>防御副使</td><td>676</td></tr><tr><td>团练副使</td><td>676</td></tr><tr><td>团副使</td><td>676</td></tr></table>",
  "<table><tr><td>副使</td><td>676</td></tr><tr><td>州别驾</td><td>676</td></tr><tr><td>别驾</td><td>676</td></tr><tr><td>州长史</td><td>676</td></tr><tr><td>长史</td><td>676</td></tr><tr><td>州司马</td><td>676</td></tr><tr><td>司马</td><td>676</td></tr><tr><td>州司士参军</td><td>676</td></tr><tr><td>司士</td><td>676</td></tr><tr><td>散参军</td><td>676</td></tr><tr><td>州文学参军</td><td>676</td></tr><tr><td>文学</td><td>676</td></tr><tr><td>州助教</td><td>676</td></tr><tr><td>助教</td><td>677</td></tr></table>"
]
"""提取表格数据后手动处理分类，保存到 json 文件中"""
refined_table_files = [
  "职官条目分类目录-table1-refined.json",
  "职官条目分类目录-table2-refined.json",
  "职官条目分类目录-table3-refined.json",
  "职官条目分类目录-table4-refined.json",
  "职官条目分类目录-table567-refined.json",
]
refined_tables = []
for filename in refined_table_files:
  with open("../../data/ocr-results/" + filename, "r", encoding="utf-8") as f:
    table = json.load(f)
  refined_tables.append(table)



In [ ]:
def get_lines_txt(lines):
  text = []
  for line in lines:
    for span in line["spans"]:
      text.append(span["content"])
  return "".join(text)

def parse_block(block, records):
  if block["type"] in ["text", "title"]:
    records.append({
      "type": block["type"],
      "text": get_lines_txt(block["lines"]),
      "bbox": block["bbox"]
    })
  elif block["type"] == "list":
    for b in block["blocks"]:
      parse_block(b, records)
  else:
    print("error block type", block["type"])

ti = 0
records = []
for page in ocr_result["pdf_info"]:
  for pb in page["para_blocks"]:
    if pb["type"] == "table":
      # 处理误提取的表格
      if ti < len(refined_tables):
        records.extend(refined_tables[ti])
      ti += 1
    else:
      parse_block(pb, records)

print(ti, 7)
print(len(records))
with open("../../data/ocr-results/职官条目分类目录-records.json", "w", encoding="utf-8") as f:
  json.dump(records, f, ensure_ascii=False, indent=2)

7 7
11406


进一步做 refine，预测结构，并手动修正
1. 根据缩进预测 surname 和 name
2. 检测其中错误项，手动修正
3. 合并多行项

In [47]:
with open("../../data/ocr-results/职官条目分类目录-records-refined.json", "r", encoding="utf-8") as f:
  records = json.load(f)
print(len(records))

"""拆分页码"""
""" 手动修复缺字（仅有页码）
倅马 39
外处拣来内品 68
复州内品 68
殿直 69
节级 77
重事 88 （错字，识别为 董事）
平章 88
平章 89
专封官 121
西省 185
太常 298
管勾北京留守司御史台公事 426 （太长导致下一行只有页码，修改并删除下一行）
内殿直左第一、第二班内殿直右第一、第二班 441（太长，且原书排版出现问题，这三个都是）
散员左第一、第二班 散员右第一、第二班 441
散指挥左第一、第二班 散指挥右第一、第二班 441
侍卫亲军司马步军副都 448 （太长）
指挥使 侍卫亲军副都指挥使 448
权签书经略安司抚判官公事 506
淮南、江、浙、荆湖制置茶盐矾税、都大发运都监 529（太长，且缺少顿号，正文中有）
淮南江浙荆湖茶盐矾税、都大发运判官 529（缺少顿号）
淮南等路制置发运司 530（和下一行连在一起了，手动分开）
江淮等路提点坑冶铸 544
安抚使司（大使司）管勾机宜文字 554（行相连）
安抚司（大使司）干办公事 555（行相连）
经制两浙、江东路 557（2行）
提领措置财用 557
权措置财用 557
倅贰 577
倅 589
倅厅 590
都尉 612
县令、录事参军 632
刺史 639
刺部 639
殿直 650
殿头 659
疏决 热敕 683
榜子 690
状元 699
叙复 724
广南西路经略安抚都总管司 508 (错版)
经略安抚制置使 508  \n经略制置使 508  \n将 508
"""

new_records = []

i = 0
while i < len(records):
  record = records[i]
  if record["type"] == "text":
    ts = record["text"].split(" ")
    if len(ts) == 2 and ts[-1].isdigit():
      assert ts[1].isdigit(), record
      new_records.append({
        "type": "text",
        "text": ts[0],
        "page": ts[1],
        "bbox": record["bbox"]
      })
      i += 1
      continue
    else:
      j = i
      text = record["text"]
      while j < len(records) and not text.split(" ")[-1].isdigit():
        j += 1
        text += records[j]["text"]
      ts = text.split(" ")
      new_records.append({
        "type": "text",
        "text": ts[0],
        "page": ts[1],
        "bbox": record["bbox"]
      })
      if len(ts) != 2:
        print(ts)
      i = j + 1
      continue
  else:
    new_records.append(record)
    i += 1

print(len(new_records))

with open("../../data/ocr-results/职官条目分类目录-refined_records.json", "w", encoding="utf-8") as f:
  json.dump(new_records, f, ensure_ascii=False, indent=2)

11470
['二丞', '贰丞', '198']
['潜火', '军兵', '267']
['内殿直左第一、第二班', '内殿直右第一、第二班', '441']
['散员左第一、第二班', '散员右第一、第二班', '441']
['散指挥左第一、第二班', '散指挥右第一、第二班', '441']
11369


In [80]:
import json
with open("../../data/ocr-results/职官条目分类目录-refined_records.json", "r", encoding="utf-8") as f:
  records = json.load(f)

"""检查标题"""
for record in records:
  if record["type"] == "title":
    # print(record)
    text = record["text"]
    if text in ["I.职官条目分类目录", "Ⅱ.职官术语与典故目录"]:
      record["type"] = "catalog"
    elif text[0] == "第":
      record["type"] = "h1"
    elif text[1] == "、" or (len(text) > 2 and text[2] == "、"):
      record["type"] = "h2"
    elif text[0] in ["附", "["]:
      record["type"] = "h2"
    else:
      record["type"] = "h3"
    print(record["type"], record["text"])

"""按照bbox的y1分列"""
columns = []
y = 0
col = []
columns.append(col)
for record in records:
  if record["type"] == "text":
    if record["bbox"][1] < y:
      y = 0
      col = []
      columns.append(col)
    
    col.append(record)
    y = record["bbox"][1]
  elif record["type"] in ["catalog", "h1"]:
    if len(col) != 0:
      y = 0
      col = []
      columns.append(col)
print(len(columns))
print(columns[0])

"""验证可行性"""
skip_cols = []
for col in columns:
  xs = set()
  for record in col:
    xs.add(record["bbox"][0])
  xs = list(xs)
  xs.sort()

  f= True
  for x in xs:
    if x - xs[0] > 5 and xs[-1] - x > 5:
      print(len(xs), xs, col[0])
      f = False
      break
  if not f:
    skip_cols.append(col)
    continue
  for x in xs:
    if xs[-1] - xs[0] > 5 and abs((x - xs[0]) - (xs[-1] - x)) < 5:
      print(len(xs), x, xs, col[0])
      f = False
      continue
  if not f:
    skip_cols.append(col)
    continue
  
  for record in col:
    x = record["bbox"][0]
    if x - xs[0] < xs[-1] - x:
      record["type"] = "name"
    else:
      record["type"] = "surname"

for record in records:
  if record["type"] not in ["title", "text"]:
    if "bbox" in record:
      del record["bbox"]

with open("../../data/ocr-results/职官条目分类目录-catalog.json", "w", encoding="utf-8") as f:
  json.dump(records, f, ensure_ascii=False, indent=2)








catalog I.职官条目分类目录
h1 第一编 皇帝制度类
h2 一、皇帝门
h2 二、后妃门
h2 三、尚书内省门
h2 四、皇太子与东宫官门
h2 五、公主与驸马都尉门
h2 六、亲王府与王府官门
h2 七、学士院门
h2 八、经筵官门
h2 九、宦官门
h2 十、翰林院等供奉机构门
h1 第二编 宰执官类
h2 一、宰相门宰相
h2 二、平章军国事门
h2 三、使相门
h2 四、执政门
h2 附：三师、三公门
h1 第三编 北宋前期中枢机构类
h2 一、中书门下门
h2 二、中书门下附属机构门
h2 三、枢密院门
h2 四、三司门
h2 五、宣徽院门
h2 六、群牧司门
h2 [附]殿阁学士与三馆秘阁门
h1 第四编 元丰正名后中枢机构类之一
h2 一、三省门
h3 门下省
h3 中书省
h3 尚书省
h2 二、三省、枢密院附属官司门
h2 三、尚书省六部门
h3 吏部
h3 户部
h3 礼部
h3 兵部
h3 刑部
h3 工部
h2 四、秘书省门
h2 [附] 修史机构
h2 五、殿中省门
h1 第五编 元丰正名后中枢机构类之二
h2 一、总九寺五监门
h2 二、太常寺门
h2 三、宗正寺大宗正司门
h2 四、光禄寺门
h2 五、卫尉寺门
h2 六、太仆寺门
h2 七、鸿胪寺门
h2 八、司农寺门
h2 九、太府寺门
h2 十、五监、国子监门
h2 十一、少府监门
h2 十二、军器监门
h2 十三、将作监门
h2 十四、都水监门
h1 第六编 司法、监察机构类
h2 一、御史台门
h2 二、谏院门
h2 三、大理寺门
h2 四、审刑院门
h2 五、刑部官门
h2 六、在京纠察刑狱司门
h1 第七编 皇宫京城禁卫侍奉机构类
h2 一、禁军三衙门
h2 二、皇城司与横行五司门
h2 三、三卫官与六统军门
h2 [附]环卫官门
h1 第八编 军事统率机构与地方治安机构类
h2 一、大元帅府、都督府门
h2 二、兵马都部署、铃辖、监押与巡检门
h2 三、制置、宣谕、招讨、经略安抚使门
h2 四、宣抚司、总领所门
h2 五、御前诸军都统制司门
h2 六、将司门
h1 第九编 地方官类之一——路官
h2 一、总监司门
h2 二、发运使、转运使门
h2 三、提点刑狱公事门
h2 四、提举常平公事门
h2 五、监、冶、场、务门
h2 六、市舶司门
h2 七、安抚使、经总

In [76]:
with open("../../data/ocr-results/职官条目分类目录-catalog-refined.json", "r", encoding="utf-8") as f:
  records = json.load(f)
for record in records:
  if "bbox" in record:
    del record["bbox"]

with open("../../data/ocr-results/职官条目分类目录-catalog-refined.json", "w", encoding="utf-8") as f:
 json.dump(records, f, ensure_ascii=False, indent=2)

outputs = []
for record in records:
  type = record["type"]
  text = record["text"]
  if type == "catalog":
    outputs.append(f"# {text}")
  elif type == "h1":
    outputs.append(f"## {text}")
  elif type == "h2":
    outputs.append(f"### {text}")
  elif type == "h3":
    outputs.append(f"#### {text}")
  elif type == "name":
    page = "{:>5}".format(record["page"])
    outputs.append(f"{page}     {text}")
  elif type == "surname":
    page = "{:>5}".format(record["page"])
    outputs.append(f"{page}          {text}")
  else:
    assert False, type
with open("../../data/ocr-results/职官条目分类目录-catalog.md", "w", encoding="utf-8") as f:
  f.write("\n".join(outputs))


In [79]:
with open("../../data/ocr-results/职官条目分类目录-catalog-refined.json", "r", encoding="utf-8") as f:
  records = json.load(f)
titles = []
for record in records:
  if record["type"] in ["h1", "h2", "h3", "catalog"]:
    titles.append(record["text"])
print(titles)







['I.职官条目分类目录', '第一编 皇帝制度类', '一、皇帝门', '二、后妃门', '三、尚书内省门', '四、皇太子与东宫官门', '五、公主与驸马都尉门', '六、亲王府与王府官门', '七、学士院门', '八、经筵官门', '九、宦官门', '十、翰林院等供奉机构门', '第二编 宰执官类', '一、宰相门宰相', '二、平章军国事门', '三、使相门', '四、执政门', '附：三师、三公门', '第三编 北宋前期中枢机构类', '一、中书门下门', '二、中书门下附属机构门', '三、枢密院门', '四、三司门', '五、宣徽院门', '六、群牧司门', '[附]殿阁学士与三馆秘阁门', '第四编 元丰正名后中枢机构类之一', '一、三省门', '门下省', '中书省', '尚书省', '二、三省、枢密院附属官司门', '三、尚书省六部门', '吏部', '户部', '礼部', '兵部', '刑部', '工部', '四、秘书省门', '[附] 修史机构', '五、殿中省门', '第五编 元丰正名后中枢机构类之二', '一、总九寺五监门', '二、太常寺门', '三、宗正寺大宗正司门', '四、光禄寺门', '五、卫尉寺门', '六、太仆寺门', '七、鸿胪寺门', '八、司农寺门', '九、太府寺门', '十、五监、国子监门', '十一、少府监门', '十二、军器监门', '十三、将作监门', '十四、都水监门', '第六编 司法、监察机构类', '一、御史台门', '二、谏院门', '三、大理寺门', '四、审刑院门', '五、刑部官门', '六、在京纠察刑狱司门', '第七编 皇宫京城禁卫侍奉机构类', '一、禁军三衙门', '二、皇城司与横行五司门', '三、三卫官与六统军门', '[附]环卫官门', '第八编 军事统率机构与地方治安机构类', '一、大元帅府、都督府门', '二、兵马都部署、钤辖、监押与巡检门', '三、制置、宣谕、招讨、经略安抚使门', '四、宣抚司、总领所门', '五、御前诸军都统制司门', '六、将司门', '第九编 地方官类之一——路官', '一、总监司门', '二、发运使、转运使门', '三、提点刑狱公事门', '四、提举常平公事门', '五、监、冶、场、务门', '六、市舶司门', '七、安抚使、经总制司门', 